In [1]:
# Import Pandas and NumPy for data handling.
import pandas as pd
import numpy as np

# Import Scikit-learn tools for encoding and scaling.
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [2]:
# Load the dataset produced during Phase 3.
file_path = "../data/processed/student_performance_featured.csv"

df = pd.read_csv(file_path)

# Inspect the dataset dimensions and first five records.
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (395, 37)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,Walc,health,absences,G1,G2,G3,Grade_Category,Attendance_Status,Study_Load,Pass_Fail
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,1,3,6,5,6,6,Low,Regular,2,Fail
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,1,3,4,5,5,6,Low,Regular,2,Fail
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,3,10,7,8,10,Medium,Regular,-1,Pass
3,GP,F,15,U,GT3,T,4,2,health,services,...,1,5,2,15,14,15,High,Regular,3,Pass
4,GP,F,16,U,GT3,T,3,3,other,other,...,2,5,4,6,10,10,Medium,Regular,2,Pass


In [3]:
# Examine the data types to identify numerical and categorical columns.
print("Column data types:")
print(df.dtypes)

# Count how many columns belong to each data type.
print("\nData type summary:")
print(df.dtypes.value_counts())

Column data types:
school                 str
sex                    str
age                  int64
address                str
famsize                str
Pstatus                str
Medu                 int64
Fedu                 int64
Mjob                   str
Fjob                   str
reason                 str
guardian               str
traveltime           int64
studytime            int64
failures             int64
schoolsup              str
famsup                 str
paid                   str
activities             str
nursery                str
higher                 str
internet               str
romantic               str
famrel               int64
freetime             int64
goout                int64
Dalc                 int64
Walc                 int64
health               int64
absences             int64
G1                   int64
G2                   int64
G3                   int64
Grade_Category         str
Attendance_Status      str
Study_Load           int64
Pass_Fail

In [4]:
# Separate the target variable from the input features.
y = df["Pass_Fail"].map({"Fail": 0, "Pass": 1})

# Remove the target and grade-derived columns to prevent target leakage.
X = df.drop(columns=["Pass_Fail", "G3", "Grade_Category"])

# Check the resulting input features and target.
print("Input features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Input features shape: (395, 34)
Target shape: (395,)

Target distribution:
Pass_Fail
1    265
0    130
Name: count, dtype: int64


In [6]:
# Identify categorical columns that require encoding.
categorical_features = X.select_dtypes(
    include=["str", "category"]
).columns.tolist()

# Identify numerical columns that require scaling.
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

# Confirm that every input feature has been classified.
assert set(categorical_features).isdisjoint(numerical_features)
assert set(categorical_features + numerical_features) == set(X.columns)

print("Categorical features:", len(categorical_features))
print(categorical_features)

print("\nNumerical features:", len(numerical_features))
print(numerical_features)

print("\nAll input features classified successfully.")

Categorical features: 18
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'Attendance_Status']

Numerical features: 16
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'Study_Load']

All input features classified successfully.


In [7]:
# Apply One-Hot Encoding to categorical features.
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Standardise numerical features.
numerical_transformer = StandardScaler()

# Combine both transformations into one preprocessing pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_features),
        ("numerical", numerical_transformer, numerical_features)
    ],
    remainder="drop"
)

print("Transformation pipeline created successfully.")

Transformation pipeline created successfully.


In [8]:
# Confirm the transformations assigned to each feature group.
print("Categorical transformation: One-Hot Encoding")
print("Numerical transformation: Standard Scaling")

print("\nNumber of categorical input features:", len(categorical_features))
print("Number of numerical input features:", len(numerical_features))

print("\nPreprocessor configuration:")
print(preprocessor)

Categorical transformation: One-Hot Encoding
Numerical transformation: Standard Scaling

Number of categorical input features: 18
Number of numerical input features: 16

Preprocessor configuration:
ColumnTransformer(transformers=[('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['school', 'sex', 'address', 'famsize',
                                  'Pstatus', 'Mjob', 'Fjob', 'reason',
                                  'guardian', 'schoolsup', 'famsup', 'paid',
                                  'activities', 'nursery', 'higher', 'internet',
                                  'romantic', 'Attendance_Status']),
                                ('numerical', StandardScaler(),
                                 ['age', 'Medu', 'Fedu', 'traveltime',
                                  'studytime', 'failures', 'famrel', 'freetime',
                          

In [9]:
# Confirm that the target variable is numeric.
print("Target data type:", y.dtype)

# Check that the input features contain the expected data types.
print("\nInput feature data types:")
print(X.dtypes.value_counts())

# Verify that the preprocessing pipeline covers every input feature.
assert len(categorical_features) + len(numerical_features) == X.shape[1]

# Confirm that target leakage columns are excluded.
assert "G3" not in X.columns
assert "Grade_Category" not in X.columns
assert "Pass_Fail" not in X.columns

print("\nTransformation setup verified successfully.")

Target data type: int64

Input feature data types:
str      18
int64    16
Name: count, dtype: int64

Transformation setup verified successfully.


## Data Transformation Summary

The dataset was prepared for machine learning by separating the target variable from the input features and identifying categorical and numerical columns.

- **Target encoding:** Pass was represented as 1 and Fail as 0.
- **Categorical encoding:** One-Hot Encoding was selected for the 18 categorical features.
- **Numerical scaling:** StandardScaler was selected for the 16 numerical features.
- **Target leakage prevention:** G3 and Grade_Category were excluded from the input features because they are derived from the final grade.

A Scikit-learn ColumnTransformer was configured to apply the appropriate transformation to each feature group.

The transformation will be fitted using the training dataset and subsequently applied to the test dataset to prevent data leakage.

The input dataset contains 395 records and 34 predictor features, with Pass_Fail maintained separately as the target variable.